# Smoke test

A tiny, no-GPU sanity check of the no-GPU core (config loading, schema, formatting, mixing,
per-mode metrics). Run with `uv run jupyter nbconvert --to notebook --execute notebooks/smoke_test.ipynb`
or open it in an editor. The GPU training/eval/export paths are exercised via the scripts in
`scripts/` (see the README run guide), not here.

In [3]:
from src.slm_coach.config import load_eval_config, load_sft_config

sft = load_sft_config("../configs/sft_multistage.yaml")
print("model:", sft.model_name, "| stages:", [s.name for s in sft.stages])

ev = load_eval_config("../configs/eval.yaml")
print("judges:", ev.judges)

ModuleNotFoundError: No module named 'src'

In [ ]:
from slm_coach.data.formatting import iter_assistant_spans, record_to_messages, render_chatml
from slm_coach.data.schema import parse_record

rec = parse_record({
    "id": "demo", "data_type": "sft", "conversation_type": "multi_turn",
    "mode": "comparison", "persona": "P01",
    "messages": [
        {"role": "user", "content": "iPhone 15 vs 14?"},
        {"role": "assistant", "content": "15 dùng USB-C."},
        {"role": "user", "content": "Gia?"},
        {"role": "assistant", "content": "~22 trieu."},
    ],
    "lang": "vi", "version": "v1", "audit_status": "approved",
})
messages = record_to_messages(rec)
text = render_chatml(messages)
spans = iter_assistant_spans(messages)
print("trainable assistant turns:", [text[s:e] for s, e in spans])

In [ ]:
from slm_coach.eval.metrics import SampleScore, aggregate_by_mode
from slm_coach.eval.rubric import CRITERIA, DEFAULT_WEIGHTS

samples = [
    SampleScore("a", "comparison", {c: 4 for c in CRITERIA}),
    SampleScore("b", "objection_handling", {c: 2 for c in CRITERIA}),
]
bd = aggregate_by_mode(samples, DEFAULT_WEIGHTS)
print("weakest:", bd.weakest_modes(1))